In [1]:
import numpy as np
import pandas as pd
import os
import plotly.express as px

In [2]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, [bias_adv]))

def calTheta(xP: np.array, theta0: np.array, alpha: float, methods: str):
    if "L1" in methods:
        return calThetaAdv_l1(xP, theta0, alpha)
    else:
        return calThetaAdv_linf(xP, theta0[:-1], theta0[-1], alpha)    

def getStats(xP: np.ndarray, x0: np.ndarray, theta: np.ndarray, lamb):
    if xP.size != theta.size:
        x0 = np.hstack((x0, 1))
        xP = np.hstack((xP, 1))

    return np.log(1 + np.exp(-(xP @ theta))) + \
        (lamb * (np.linalg.norm(x0 - xP, ord=1)))

In [6]:
def readPickle(final_path: str):
    df = pd.read_pickle(final_path)
    df["theta_r"] = df.apply(lambda row : calTheta(row['x_r'], row['theta_0'], row['alpha'], row['algorithm']), axis=1)
    df['J'] = df.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)

    return df

def readPickleAll(dir_path: str, clf_name:str, dataset_name: str, lambda_list: list, alpha_list:list):
    all_files = []

    for file in os.listdir(dir_path):
        splitted_file = file.split(sep='_')
        if dataset_name in splitted_file and clf_name in splitted_file and float(splitted_file[3]) in lambda_list and float(splitted_file[4]) in alpha_list:
            all_files.append(os.path.join(dir_path, file))
            
    df = pd.concat([pd.read_pickle(f_n) for f_n in all_files])

    df["theta_r"] = df.apply(lambda row : calTheta(row['x_r'], row['theta_0'], row['alpha'], row['algorithm']), axis=1)
    df['J'] = df.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)
    df = df.groupby("algorithm", group_keys=False).apply(lambda x : x.reset_index(drop=True))

    return df

In [72]:
dir_path = "../results/recourse"
clf_name = "lr"
dataset_name = "synthetic"
lambdas = [0.1,0,2]
alphas = [0.02, 0.1, 0.2]

output = readPickleAll(dir_path, clf_name, dataset_name, lambdas, alphas)
# mask = output['algorithm'] == 'ROAR'
# output.loc[mask, 'algorithm'] = 'ROARLInf'
# mask = output['algorithm'] == 'Alg1'
# output.loc[mask, 'algorithm'] = 'LInf'

/var/folders/pc/zqw2f9xx69vcd5v3_xxyvr84slyzh9/T/ipykernel_12775/2730404693.py:20: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [73]:
mask = output['algorithm'] == 'Alg1'
output[mask]



,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0,theta_r,J
0,Alg1,2,0.20,0.1,0,"[-2.5687, -2.205]","[1.6693, 0.0]","[1.9521, 1.9468, 0.0798]","[1.7521, 1.7468000000000001, -0.12020000000000...",0.703071
1,Alg1,2,0.20,0.1,1,"[-2.1753, -3.0077]","[1.6693, 0.0]","[1.9521, 1.9468, 0.0798]","[1.7521, 1.7468000000000001, -0.12020000000000...",0.744001
2,Alg1,2,0.20,0.1,2,"[-1.0416, -1.9984]","[1.6693, 0.0]","[1.9521, 1.9468, 0.0798]","[1.7521, 1.7468000000000001, -0.12020000000000...",0.529701
3,Alg1,2,0.20,0.1,3,"[-1.8496, -1.4463]","[1.6693, 0.0]","[1.9521, 1.9468, 0.0798]","[1.7521, 1.7468000000000001, -0.12020000000000...",0.555291
4,Alg1,2,0.20,0.1,4,"[-1.403, -2.0718]","[1.6693, 0.0]","[1.9521, 1.9468, 0.0798]","[1.7521, 1.7468000000000001, -0.12020000000000...",0.573181
...,...,...,...,...,...,...,...,...,...,...
1495,Alg1,4,0.02,0.1,100,"[-2.8384, -3.6958]","[5.0922, -3.6958]","[1.9853, 1.9103, 0.0723]","[1.9653, 1.9303000000000001, 0.0523]",0.845283
1496,Alg1,4,0.02,0.1,101,"[-1.7577, -2.538]","[3.955, -2.538]","[1.9853, 1.9103, 0.0723]","[1.9653, 1.9303000000000001, 0.0523]",0.623495
1497,Alg1,4,0.02,0.1,102,"[-2.0031, -1.5358]","[2.9707, -1.5358]","[1.9853, 1.9103, 0.0723]","[1.9653, 1.9303000000000001, 0.0523]",0.549600
1498,Alg1,4,0.02,0.1,103,"[-3.0075, -0.669]","[2.1193, -0.669]","[1.9853, 1.9103, 0.0723]","[1.9653, 1.9303000000000001, 0.0523]",0.564904


In [92]:
lamb = 0.1

output_mean = output.groupby(['algorithm', 'alpha', 'lambda'], as_index=False).mean()
# px.scatter(output, y="J", color="algorithm", facet_col="algorithm")
mask = output_mean['lambda'] == lamb
output_mean = output_mean[mask]
output_mean

,algorithm,alpha,lambda,seed,i,x_0,x_r,theta_0,theta_r,J
0,Alg1,0.000,0.1,1.989418,18.433862,"[-0.00016455026455024422, -0.27871957671957703...","[-0.00016455026455024422, -0.27871957671957703...","[0.27284391534391456, -0.17822169312169434, -0...","[0.27284391534391456, -0.17822169312169434, -0...",0.269301
2,Alg1,0.125,0.1,1.989418,18.433862,"[-0.00016455026455024422, -0.27871957671957703...","[-0.00016455026455024422, -0.27871957671957703...","[0.27284391534391456, -0.17822169312169434, -0...","[0.28673280423280345, -0.1405232804232809, -0....",0.328266
4,Alg1,0.250,0.1,1.989418,18.433862,"[-0.00016455026455024422, -0.27871957671957703...","[-0.00016455026455024422, -0.27871957671957703...","[0.27284391534391456, -0.17822169312169434, -0...","[0.3006216931216924, -0.10282486772486808, -0....",0.391961
6,Alg1,0.375,0.1,1.989418,18.433862,"[-0.00016455026455024422, -0.27871957671957703...","[-0.00016455026455024422, -0.27871957671957703...","[0.27284391534391456, -0.17822169312169434, -0...","[0.3145105820105813, -0.06512645502645496, -0....",0.461065
8,Alg1,0.500,0.1,1.989418,18.433862,"[-0.00016455026455024422, -0.27871957671957703...","[-0.00016455026455024422, -0.27871957671957703...","[0.27284391534391456, -0.17822169312169434, -0...","[0.32839947089947014, -0.027428042328042204, -...",0.536269
10,L1PSD,0.000,0.1,2.000000,2.321429,"[-0.30137857142857155, -0.41843571428571436, 0...","[-0.2994178571428571, -0.4175892857142857, 0.5...","[0.27531428571428573, -0.17767142857142848, -0...","[0.27531428571428573, -0.17767142857142848, -0...",0.290356
12,L1PSD,0.125,0.1,2.000000,2.321429,"[-0.30137857142857155, -0.41843571428571436, 0...","[-0.3014107142857143, -0.4193321428571428, 0.5...","[0.27531428571428573, -0.17767142857142848, -0...","[0.27085, -0.17767142857142848, -0.00629642857...",0.298336
14,L1PSD,0.250,0.1,2.000000,2.321429,"[-0.30137857142857155, -0.41843571428571436, 0...","[-0.30247857142857143, -0.41945, 0.56918571428...","[0.27531428571428573, -0.17767142857142848, -0...","[0.2663857142857143, -0.17767142857142848, -0....",0.305606
16,L1PSD,0.375,0.1,2.000000,2.321429,"[-0.30137857142857155, -0.41843571428571436, 0...","[-0.3023321428571429, -0.4189642857142856, 0.5...","[0.27531428571428573, -0.17767142857142848, -0...","[0.2619214285714286, -0.17767142857142848, -0....",0.312952
18,L1PSD,0.500,0.1,2.000000,2.321429,"[-0.30137857142857155, -0.41843571428571436, 0...","[-0.3019428571428572, -0.41958571428571423, 0....","[0.27531428571428573, -0.17767142857142848, -0...","[0.2574571428571429, -0.17767142857142848, -0....",0.320024


In [93]:
custom_colors = {
    "LInf": "#33FFFF",
    "L1PSD": "#FF3333",
    "ROARL1": "#33FF33",
    "ROARLInf": "#FF33FF",
}


fig = px.line(output_mean, x = "alpha", y = "J", color="algorithm", 
           labels = {"seed" : "Fold", "J" : "Total Cost"},
           title=f"{clf_name}_{dataset_name}_lambda{lamb}_costMean",
            markers=True,
            color_discrete_map=custom_colors )
fig

# fig.write_html(f"{clf_name}_{dataset_name}_lambda{lamb}_costMean.html")
# fig

In [ ]:
# rng = np.random.default_rng(seed=seed)
# size_N = int(np.rint(0.15 * recourse_needed_X_test.shape[0]))
# recourse_needed_X_test = rng.choice(recourse_needed_X_test, size=size_N, replace=False)

output.groupby(['lambda', 'alpha', 'algorithm', 'seed']).count().apply(lambda row: np.arange(0,row['i']),axis=1)

lambda  alpha  algorithm  seed
0.1     0.0    Alg1       0       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          1       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          2       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          3       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          4       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                                                        ...                        
0.3     0.5    ROARLInf   0       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          1       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          2       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          3       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          4       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
Length: 200, dtype: object

In [70]:
# Comparsion between which feature changes

diff_value = 1e-3

output["cost"] = np.linalg.norm(np.stack(output["x_0"].values) - np.stack(output["x_r"].values), 1, 1)
output['diff'] = output.apply(lambda row : np.abs(row['x_r'] - row['x_0']), axis=1)
output['diff_bool'] = output['diff'].apply(lambda x: np.where(x > diff_value, 1, 0))
# output.sort_values(by=["seed"]).sort_index()

# output_with_i = output.copy()
# output_with_i["row_index"] = output.index
# output_diff = output_with_i.sort_values(["seed", "row_index", "algorithm"]).drop(columns="row_index")

output = output.reset_index(drop=True)
output['sparsity'] = output['diff_bool'].apply(lambda x: sum(x)).reset_index(drop=True)
# output[output['sparsity'] == 25]

print(np.stack(output['theta_0'].abs()).min(axis=1))


# output['cost'].describe()
# output_diff['diff_bool'].apply(lambda x: sum(x))
px.scatter(output, y="cost", x="sparsity", color="algorithm")


[0.0798 0.0798 0.0798 ... 0.0723 0.0723 0.0723]


In [123]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_L1PSD_1.pkl"
final_path = os.path.join(dir_path, file_name)

# df2 = pd.read_pickle(final_path)
# # df2 = df2.rename(columns={"x_r": "theta_0", "theta_0": "x_r"})
# df2['theta_r'] = df2.apply(lambda row : calThetaAdv_l1(np.hstack((row['x_r'], 1)), row['theta_0'], row['alpha']), axis=1)
# df2['J'] = df2.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)
# # df2.loc[:,['algorithm', 'seed', 'alpha', 'lambda', 'i', 'x_0', 'x_r', 'theta_0']]

# # df2.to_pickle(final_path)

# df2

df2 = readPickle(final_path)

In [ ]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_ROAR_0.pkl"
final_path = os.path.join(dir_path, file_name)

df3 = pd.read_pickle(final_path)